#  Import Libraries

In [8]:
import copy
import itertools
import warnings
import os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.nn import GCNConv, MessagePassing
from torch_geometric.utils import add_self_loops

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
import numpy as np
from imblearn.over_sampling import SMOTE

from tqdm import tqdm, trange

import matplotlib.pyplot as plt

print('Imports OK')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports OK
PyTorch  : 2.6.0+cu124
CUDA available: True


#  Parse Raw Data and Create Kaggle Format Dataset

In [9]:
# Parse the raw semicolon-delimited file
raw_df = pd.read_csv('LUSCexpfile.csv')
df = raw_df.iloc[:, 0].str.split(';', expand=True)
labels_raw = df.iloc[0, 1:]          # first row = sample labels
genes = df.iloc[1:, 0]               # first col = gene names
expression = df.iloc[1:, 1:]
expression.index = genes
expression = expression.apply(pd.to_numeric, errors='coerce')
expression_t = expression.T          # samples × genes

# Build binary labels
labels_clean = (
    labels_raw
    .astype(str)
    .str.lower()
    .str.strip()
    .replace('none', pd.NA)
)
binary_labels = labels_clean.apply(
    lambda x: 1 if isinstance(x, str) and 'tumor' in x
         else 0 if isinstance(x, str) and 'normal' in x
         else pd.NA
)
valid_idx = binary_labels.notna()
binary_labels = binary_labels[valid_idx].astype(int)
expression_t = expression_t.loc[valid_idx]
expression_t['label'] = binary_labels.values
expression_t.insert(0, 'sample_id', expression_t.index)
expression_t.reset_index(drop=True, inplace=True)
expression_t.to_csv('TCGA_LUSC_Kaggle_Format.csv', index=False)
print('Saved TCGA_LUSC_Kaggle_Format.csv')


Saved TCGA_LUSC_Kaggle_Format.csv


# Load and Explore Data

In [10]:
data1 = pd.read_csv("TCGA_LUSC_Kaggle_Format.csv")

print("Columns:", data1.columns.tolist()[:10], "...")  # Show first 10 columns
print("\nLabel value counts:")
print(data1["label"].value_counts())


Columns: ['sample_id', 'A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2ML1-AS1', 'A2ML1-AS2', 'A2MP1'] ...

Label value counts:
label
1    502
0     49
Name: count, dtype: int64


#  Apply SMOTE and Create Graph

In [11]:
# Build feature tensors
df_lusc = pd.read_csv('TCGA_LUSC_Kaggle_Format.csv')
X = df_lusc.drop(columns=['sample_id', 'label']).values
y = df_lusc['label'].values

# Remove zero-variance features
var = X.var(axis=0)
X = X[:, var > 0]
X = np.nan_to_num(X)

# Standardize BEFORE SMOTE
X = StandardScaler().fit_transform(X)

print(f'Before SMOTE:')
print(f'  Class 0 (normal): {np.sum(y == 0)}')
print(f'  Class 1 (tumor) : {np.sum(y == 1)}')

# Apply SMOTE to balance classes
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f'\nAfter SMOTE:')
print(f'  Class 0 (normal): {np.sum(y_resampled == 0)}')
print(f'  Class 1 (tumor) : {np.sum(y_resampled == 1)}')

# Update X and y with resampled data
X = X_resampled
y = y_resampled

# Save SMOTE-balanced dataset
feature_columns = df_lusc.drop(columns=['sample_id', 'label']).columns[var > 0]

smote_df = pd.DataFrame(X_resampled, columns=feature_columns)
smote_df['label'] = y_resampled

smote_df.to_csv('LUSC_smote.csv', index=False)

print('\nSaved SMOTE-balanced dataset as LUSC_smote.csv')

# Split features into K groups
K = 4
gene_splits = np.array_split(np.arange(X.shape[1]), K)
X_list = [torch.tensor(X[:, idx], dtype=torch.float32) for idx in gene_splits]

# Patient kNN graph
k = 10
sim = cosine_similarity(X)
np.fill_diagonal(sim, 0)

edges = []
weights = []
for i in range(len(X)):
    topk_idx = np.argsort(sim[i])[-k:]
    for j in topk_idx:
        edges.append([i, j])
        weights.append(sim[i, j])

edge_index = torch.tensor(edges, dtype=torch.long).T
edge_weight = torch.tensor(weights, dtype=torch.float32)

print(f'\nPatient graph: {len(X)} nodes, {edge_index.size(1)} edges')


Before SMOTE:
  Class 0 (normal): 49
  Class 1 (tumor) : 502

After SMOTE:
  Class 0 (normal): 502
  Class 1 (tumor) : 502

Saved SMOTE-balanced dataset as LUSC_smote.csv

Patient graph: 1004 nodes, 10040 edges


#  Define FocalLoss

In [12]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, logits, labels):
        p = torch.sigmoid(logits).view(-1)
        labels = labels.float().view(-1)
        ce = F.binary_cross_entropy_with_logits(logits.view(-1), labels, reduction='none')
        p_t = p * labels + (1 - p) * (1 - labels)
        alpha_t = self.alpha * labels + (1 - self.alpha) * (1 - labels)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = (focal_weight * ce).mean()
        return loss


#  Model Definition

In [13]:
def fuzzy_covering(adj, k=5):
    """
    Create fuzzy coverings from adjacency matrix.
    Returns list of k-nearest neighbors with their similarity weights.
    """
    coverings = []
    for i in range(adj.shape[0]):
        # Get top-k neighbors (excluding self)
        topk_idx = np.argsort(adj[i])[-k:]
        covering = [(int(topk_idx[j]), float(adj[i, topk_idx[j]])) for j in range(k)]
        coverings.append(covering)
    return coverings

def coverings_to_edge_index(coverings, return_weights=True):
    """
    Convert fuzzy coverings to PyTorch Geometric edge format.
    """
    edges = []
    weights = []
    
    for i, covering in enumerate(coverings):
        for neighbor, weight in covering:
            edges.append([i, neighbor])
            weights.append(weight)
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    
    if return_weights:
        edge_weight = torch.tensor(weights, dtype=torch.float32)
        return edge_index, edge_weight
    return edge_index

def build_fuzzy_graph(X, k=10):
    """
    Build fuzzy graph from feature matrix using cosine similarity.
    
    Args:
        X: Feature matrix [num_samples, num_features] (numpy array or torch tensor)
        k: Number of nearest neighbors for fuzzy covering
    
    Returns:
        edge_index: Edge indices [2, num_edges]
        edge_weight: Fuzzy membership weights [num_edges]
        coverings: List of fuzzy coverings
    """
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Convert to numpy if torch tensor
    if isinstance(X, torch.Tensor):
        X = X.numpy()
    
    # Compute similarity matrix
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    
    # Create fuzzy coverings
    coverings = fuzzy_covering(sim, k=k)
    
    # Convert to edge format
    edge_index, edge_weight = coverings_to_edge_index(coverings)
    
    return edge_index, edge_weight, coverings

# Fuzzy Graph Convolution Layer 
class FuzzyCoverConv(MessagePassing):
    """
    Fuzzy covering-based graph convolution with learnable aggregation.
    """
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.dropout = dropout
        
        # Linear transformation
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        
        # Attention mechanism for fuzzy weighting
        self.att_src = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, out_channels))
        
        self.bias = nn.Parameter(torch.Tensor(out_channels))
        
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, edge_index, edge_weight=None):
        """
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: Edge indices [2, num_edges]
            edge_weight: Fuzzy membership weights [num_edges]
        """
        # Linear transformation
        x = self.lin(x).view(-1, self.heads, self.out_channels)
        
        # Propagate messages
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)
        
        # Average over heads
        out = out.mean(dim=1)
        out = out + self.bias
        
        return out
    
    def message(self, x_i, x_j, edge_weight, index, ptr, size_i):
        """
        Compute messages with fuzzy attention.
        """
        # Compute attention scores
        alpha = (x_i * self.att_src).sum(dim=-1) + (x_j * self.att_dst).sum(dim=-1)
        alpha = F.leaky_relu(alpha, 0.2)
        
        # Incorporate fuzzy edge weights
        if edge_weight is not None:
            alpha = alpha * edge_weight.view(-1, 1)
        
        # Softmax normalization
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        
        # Weighted message
        return x_j * alpha.unsqueeze(-1)

# Fuzzy Graph Convolutional Network 
class FGC_GNN(nn.Module):
    """
    Fuzzy Graph Convolutional Network for multi-omics classification.
    Integrates fuzzy covering-based convolutions with attention mechanisms.
    
    This model is dataset-independent and can be used with any preprocessed data.
    """
    def __init__(self, in_dims, hidden_channels, out_channels, 
                 heads=4, dropout=0.3, use_fuzzy_conv=True):
        """
        Args:
            in_dims: List of input dimensions for each omics layer
            hidden_channels: Hidden dimension size
            out_channels: Number of output classes
            heads: Number of attention heads
            dropout: Dropout probability
            use_fuzzy_conv: Use FuzzyCoverConv if True, otherwise GCNConv
        """
        super().__init__()
        self.dropout_p = dropout
        self.use_fuzzy_conv = use_fuzzy_conv
        self.num_omics = len(in_dims)
        
        # Per-omics graph convolution branches
        if use_fuzzy_conv:
            self.omics_gcns = nn.ModuleList([
                FuzzyCoverConv(d, hidden_channels, heads=heads, dropout=dropout) 
                for d in in_dims
            ])
        else:
            from torch_geometric.nn import GCNConv
            self.omics_gcns = nn.ModuleList([
                GCNConv(d, hidden_channels, improved=True) 
                for d in in_dims
            ])
        
        # Residual projections
        self.res_proj = nn.ModuleList([
            nn.Linear(d, hidden_channels) for d in in_dims
        ])
        
        # Fuzzy attention gates per omics branch
        self.attn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels // 4),
                nn.ReLU(),
                nn.Linear(hidden_channels // 4, 1),
                nn.Sigmoid()
            ) for _ in range(self.num_omics)
        ])
        
        # Cross-omics attention
        self.cross_attn = nn.MultiheadAttention(
            hidden_channels, num_heads=heads, dropout=dropout, batch_first=True
        )
        
        # Fusion layers
        self.fc1 = nn.Linear(hidden_channels * self.num_omics, hidden_channels * 2)
        self.bn1 = nn.BatchNorm1d(hidden_channels * 2)
        
        self.fc2 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        
        self.fc3 = nn.Linear(hidden_channels, out_channels)
        
        # Layer normalization for each omics
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_channels) for _ in range(self.num_omics)
        ])
    
    def forward(self, X_list, edge_index, edge_weight=None):
        """
        Args:
            X_list: List of feature tensors, one per omics [num_nodes, features_i]
            edge_index: Edge indices from fuzzy covering [2, num_edges]
            edge_weight: Fuzzy membership weights [num_edges]
        
        Returns:
            out: Classification logits [num_nodes, out_channels]
        """
        omics_embeddings = []
        
        # Process each omics layer
        for i, (gcn, res, attn, ln, X) in enumerate(
            zip(self.omics_gcns, self.res_proj, self.attn, self.layer_norms, X_list)
        ):
            # Graph convolution with fuzzy weights
            h = gcn(X, edge_index, edge_weight=edge_weight)
            
            # Residual connection
            h_res = res(X)
            h = h + h_res
            
            # Layer normalization
            h = ln(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout_p, training=self.training)
            
            # Fuzzy attention gate
            attn_weight = attn(h)
            h = attn_weight * h
            
            omics_embeddings.append(h)
        
        # Stack for cross-omics attention [num_nodes, num_omics, hidden_channels]
        omics_stack = torch.stack(omics_embeddings, dim=1)
        
        # Apply cross-omics attention
        attn_out, _ = self.cross_attn(omics_stack, omics_stack, omics_stack)
        
        # Residual connection for attention
        omics_stack = omics_stack + attn_out
        
        # Flatten for fusion
        h = omics_stack.reshape(omics_stack.size(0), -1)
        
        # Multi-layer fusion with residual
        h1 = F.relu(self.bn1(self.fc1(h)))
        h1 = F.dropout(h1, p=self.dropout_p, training=self.training)
        
        h2 = F.relu(self.bn2(self.fc2(h1)))
        h2 = F.dropout(h2, p=self.dropout_p, training=self.training)
        
        # Output layer
        out = self.fc3(h2)
        
        return out

#  Define EarlyStopping Class

In [14]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.best_epoch = -1
        self.best_state = None
    
    def step(self, score, model, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        
        improved = (
            (self.mode == 'max' and score > self.best_score + self.min_delta) or
            (self.mode == 'min' and score < self.best_score - self.min_delta)
        )
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            return False
    
    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


#  Define Utility Functions

In [15]:
def gpu_metrics(probs, labels, thr=None):
    """
    Compute metrics on GPU tensors.
    If thr is None, find optimal threshold using Youden's J.
    Returns: (auc, f1, acc, precision, recall, threshold)
    """
    probs_cpu = probs.cpu().numpy()
    labels_cpu = labels.cpu().numpy()
    
    if len(np.unique(labels_cpu)) < 2:
        return 0.5, 0.0, 0.0, 0.0, 0.0, 0.5
    
    auc = roc_auc_score(labels_cpu, probs_cpu)
    
    if thr is None:
        # Find optimal threshold using Youden's J
        from sklearn.metrics import roc_curve
        fpr, tpr, thresholds = roc_curve(labels_cpu, probs_cpu)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        thr = thresholds[best_idx]
    
    preds = (probs_cpu >= thr).astype(int)
    f1 = f1_score(labels_cpu, preds, zero_division=0)
    acc = accuracy_score(labels_cpu, preds)
    
    from sklearn.metrics import precision_score, recall_score
    prec = precision_score(labels_cpu, preds, zero_division=0)
    rec = recall_score(labels_cpu, preds, zero_division=0)
    
    return auc, f1, acc, prec, rec, thr


#  Setup Device and Output Directory

In [16]:
DATASET = 'LUSC'
OUT_DIR = 'LUSC_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

IS_CUDA = torch.cuda.is_available()
device = torch.device('cuda' if IS_CUDA else 'cpu')
print(f'Device: {device}')

# Move data to GPU
X_list_gpu = [x.to(device) for x in X_list]
edge_idx_gpu = edge_index.to(device)
edge_wt_gpu = edge_weight.to(device)
y_gpu = torch.tensor(y, dtype=torch.float32, device=device)

in_dims_cv = [x.size(1) for x in X_list_gpu]
print(f'Feature group dimensions: {in_dims_cv}')


Device: cuda
Feature group dimensions: [13815, 13815, 13815, 13814]


#  Define Hyperparameter Grid

In [17]:
PARAM_GRID = {
    'hidden_channels': [128, 256, 512],
    'lr'             : [1e-3, 2e-4],
    'weight_decay'   : [1e-4, 1e-5],
    'dropout'        : [0.3, 0.5],
    'focal_gamma'    : [1, 2],
    'focal_alpha'    : [0.5, 0.75],
    'epochs'         : [400, 700, 1000],
}

# Generate all combinations
param_combinations = list(itertools.product(*PARAM_GRID.values()))
print(f'Total hyperparameter combinations: {len(param_combinations)}')

Total hyperparameter combinations: 288


#  Train/Test Split (80/20)

In [18]:
train_idx_np, test_idx_np, y_train_np, y_test_np = train_test_split(
    np.arange(len(y)),
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train samples: {len(train_idx_np)} ({len(train_idx_np)/len(y)*100:.1f}%)')
print(f'Test samples: {len(test_idx_np)} ({len(test_idx_np)/len(y)*100:.1f}%)')


Train samples: 803 (80.0%)
Test samples: 201 (20.0%)



# 10-Run Repeated Test Evaluation using saved .pt checkpoint



In [20]:

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score
)



#  Config 
DATASET   = 'LUSC'
OUT_DIR   = 'LUSC_outputs'
CKPT_PATH = f'{OUT_DIR}/FGC_GNN_{DATASET}_best_final.pt'
N_RUNS    = 10
TEST_SIZE = 0.20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Rebuild model with best HPs 

summary_df = pd.read_csv(f'{OUT_DIR}/FGC_GNN_{DATASET}_summary.csv')
BEST_THR          = float(summary_df['Threshold'].iloc[0])
best_hidden       = int(summary_df['HP_hidden_channels'].iloc[0])
best_dropout      = float(summary_df['HP_dropout'].iloc[0])

print(f'Best threshold : {BEST_THR:.4f}')
print(f'Hidden channels: {best_hidden}')
print(f'Dropout        : {best_dropout}')

#  Load checkpoint once 
def load_model(ckpt_path, in_dims, hidden_channels, dropout, device):
    model = FGC_GNN(
        in_dims=in_dims,
        hidden_channels=hidden_channels,
        out_channels=1,
        dropout=dropout,
    ).to(device)
    state = torch.load(ckpt_path, map_location=device)
    # Strip torch.compile prefix if present
    clean_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    model.load_state_dict(clean_state)
    model.eval()
    return model

#  Assumes X_list_gpu, edge_idx_gpu, edge_wt_gpu, y, y_gpu are already in memory 

in_dims = [x.size(1) for x in X_list_gpu]

#  10-Run Evaluation 
results = []

for run in range(N_RUNS):
    # Different random seed each run for 20% test split
    seed = 42 + run

    _, test_idx, _, _ = train_test_split(
        np.arange(len(y)),
        y,
        test_size=TEST_SIZE,
        random_state=seed,
        stratify=y
    )

    test_t    = torch.tensor(test_idx, dtype=torch.long, device=device)
    y_te_gpu  = y_gpu[test_t]

    # Load fresh copy of the model
    model = load_model(CKPT_PATH, in_dims, best_hidden, best_dropout, device)

    with torch.inference_mode():
        out = model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
        te_probs = out[test_t].view(-1)

    auc, f1, acc, prec, rec, _ = gpu_metrics(te_probs, y_te_gpu, thr=BEST_THR)

    results.append({
        'Run'      : run + 1,
        'Seed'     : seed,
        'AUC'      : auc,
        'F1'       : f1,
        'Accuracy' : acc,
        'Precision': prec,
        'Recall'   : rec,
    })

    print(f'Run {run+1:2d} | AUC={auc:.4f}  F1={f1:.4f}  '
          f'Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}')

# Summary 
results_df = pd.DataFrame(results)

metrics = ['AUC', 'F1', 'Accuracy', 'Precision', 'Recall']
summary = results_df[metrics].agg(['mean', 'std'])

print(f'\n{"="*60}')
print(f'  10-Run Test Results ({TEST_SIZE*100:.0f}% split each run) — {DATASET}')
print(f'{"="*60}')
for m in metrics:
    mu  = summary.loc['mean', m]
    std = summary.loc['std',  m]
    print(f'  {m:<12}: {mu:.4f} ± {std:.4f}')
print(f'{"="*60}')

# Save to CSV 
out_csv = f'{OUT_DIR}/FGC_GNN_{DATASET}_10run_test_results.csv'
results_df.to_csv(out_csv, index=False)

summary_row = {m: f"{summary.loc['mean',m]:.4f} ± {summary.loc['std',m]:.4f}"
               for m in metrics}
summary_row['Run'] = 'MEAN ± STD'
pd.concat([results_df, pd.DataFrame([summary_row])], ignore_index=True)\
  .to_csv(out_csv, index=False)

print(f'\nResults saved → {out_csv}')

Device: cuda
Best threshold : -1.6125
Hidden channels: 512
Dropout        : 0.3
Run  1 | AUC=1.0000  F1=0.9899  Acc=0.9900  Prec=1.0000  Rec=0.9800
Run  2 | AUC=1.0000  F1=0.9899  Acc=0.9900  Prec=1.0000  Rec=0.9800
Run  3 | AUC=1.0000  F1=1.0000  Acc=1.0000  Prec=1.0000  Rec=1.0000
Run  4 | AUC=1.0000  F1=1.0000  Acc=1.0000  Prec=1.0000  Rec=1.0000
Run  5 | AUC=1.0000  F1=0.9950  Acc=0.9950  Prec=1.0000  Rec=0.9901
Run  6 | AUC=1.0000  F1=1.0000  Acc=1.0000  Prec=1.0000  Rec=1.0000
Run  7 | AUC=1.0000  F1=1.0000  Acc=1.0000  Prec=1.0000  Rec=1.0000
Run  8 | AUC=1.0000  F1=0.9950  Acc=0.9950  Prec=1.0000  Rec=0.9900
Run  9 | AUC=1.0000  F1=1.0000  Acc=1.0000  Prec=1.0000  Rec=1.0000
Run 10 | AUC=1.0000  F1=0.9950  Acc=0.9950  Prec=1.0000  Rec=0.9900

  10-Run Test Results (20% split each run) — LUSC
  AUC         : 1.0000 ± 0.0000
  F1          : 0.9965 ± 0.0042
  Accuracy    : 0.9965 ± 0.0041
  Precision   : 1.0000 ± 0.0000
  Recall      : 0.9930 ± 0.0082

Results saved → LUSC_outputs